# Project 1 – To Do List



## Problem Definition

1. Write a concise problem definition for the project. Put it in a text field at the top of your Jupyter notebook.



The goal of this project is to predict whether a customer will make a future transaction based on historical transaction data. This project will use Gaussian Naive Bayes classification as the machine learning model.

## Data Collection





###Import Data


2. Load Pandas, Numpy, and Matplotlib.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets, metrics, model_selection
from sklearn import model_selection
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

3. Load data Train.csv from AWS S3.

In [ ]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.1-Transactions/Data/Transaction.train.big.csv'
url

In [ ]:
df = pd.read_csv(url)
df

## Data Cleaning

4. Examine the data using tools we have used in class.



###Make a copy

In [ ]:
df_clean = df.copy()
df_clean

###shape

In [ ]:
df_clean.shape

###isnull()

In [ ]:
df_clean.isnull().sum()*1000

In [ ]:
df_clean.isnull().sum().sort_values()

###value_counts()

In [ ]:
df_clean['target'].value_counts()

In [ ]:
#dropna = False includes how many nulls as well
df_clean['target'].value_counts(dropna = False)

5. If there are data cleaning issues, develop recommendations for how to deal with them.

###dropna()

####Rows


In [ ]:
#Checks for nulls in the specified column and deletes that row.
df_clean.dropna(subset = ['target'])

In [ ]:
#Verify nothing has changed by running shape. Row count remained the same
df_clean.shape

In [ ]:
#inplace = True changes data. Note rows at bottom right
df_clean.dropna(subset = ['target'], inplace = True)
df_clean

In [ ]:
#Run shape to verify
df_clean.shape

####Columns

In [ ]:
df_clean.columns

In [ ]:
#Just to view. Watch how the columns in shape 104 change to 53 in this dropna
df_clean.dropna(axis='columns')

In [ ]:
#View dropped column nulls
df_clean.dropna(axis='columns', how = 'all')

In [ ]:
#Make actual change
df_clean.dropna(axis='columns', how = 'all', inplace = True)

In [ ]:
#Verify rows and columns
df_clean.shape

In [ ]:
df_clean.isnull().sum().sum()

In [ ]:
df_backup = df_clean.copy()

## Exploratory Data Analysis








6. Produce some visual analysis of the data – like plots showing the distributions of all variables. Recall that Gaussian Naive Bayes assumes the predictors are normally distributed. Note: you might have to do multiple plots in groups.


###hist()

In [ ]:
df_backup.shape

In [ ]:
#value_counts() counts how many times each value appears.
df_backup['target'].value_counts().rename(index={
    0: 'No Transaction',
    1: 'Transaction'
}).plot(kind='bar', color='pink')

plt.title("Binary Target Distribution")
plt.xlabel('Number of Observations')
plt.ylabel('Target')
plt.xticks(rotation=0)  #Makes labels horizontal

plt.tight_layout()
plt.savefig('Binary Target.png')
plt.show()

In [ ]:
df_backup.columns

In [ ]:
#bin
n = int(180_000**(1/2))

plt.figure(figsize=(10,6))


plt.hist(df_backup['var_7'], alpha=0.5, bins=n, label='var_7')
plt.hist(df_backup['var_57'], alpha=0.5, bins=n, label='var_57')
plt.hist(df_backup['var_92'], alpha=0.5, bins=n, label='var_92')

plt.legend()
plt.title("Distribution of Predictor Variables")
plt.xlabel("Predictors")
plt.ylabel("Observations")

plt.savefig('Predictor Vars.png')
plt.show()

In [ ]:
!ls

7. NOTE: the ‘target’ column indicates a successful transaction (‘1’) or a no-transaction (‘0’). Verify these are the only values in that column.



In [ ]:
df_backup['target'].value_counts(dropna = False)

In [ ]:
# df_clean.drop(columns=['Unnamed: 0', 'target'], inplace=True)
# df_clean

8. Check the correlation values between all **predictor columns** to ensure there are no substantial correlations between predictors. This is important to support the decision to classify the ‘target’ using Naïve Bayes.



###corr()

In [ ]:
import seaborn as sns

In [ ]:
#Should always have a diagonal of 1.0
corr = df_backup.drop(columns=['Unnamed: 0', 'ID_code', 'target']).corr() #Actual correlation matrix

plt.figure(figsize=(8, 5))

#annot prints the actual numbers inside each box
sns.heatmap(corr, cmap='BuPu', vmin=-1, vmax=1)
plt.title('Correlation Between Predictor Variables')
plt.xlabel('Predictor Variables (Features)')
plt.ylabel('Predictor Variables (Features)')

plt.savefig('Correlation Matrix.png')
plt.show()

In [ ]:
df_clean.shape

9. Create two data frames: one with all successful transactions, one with all unsuccessful transactions. **Make sure they are copies and not slices**.

In [ ]:
successful = df_backup[df_backup['target'] == 1].copy()
successful

In [ ]:
unsuccessful = df_backup[df_backup['target'] == 0].copy()
unsuccessful

## Data Processing





10. Create two data frames: one with all the predictor columns (everything except for Unnamed: 0, ID_code and target) and one with just the target. Make sure they are copies and not slices.



In [ ]:
X = df_backup.drop(columns=['Unnamed: 0', 'ID_code', 'target']).copy()
X

In [ ]:
y = df_backup['target'].copy()
y

11. Define a Gaussian Naïve Bayes model using Sklearn.



In [ ]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()

12. Divide the two data frames you created in step #10 into training and testing subsets.



In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)
[ _.shape for _ in [X_train, X_test, y_train, y_test] ]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42 #Means the same split every time
)

13. Train the model using the training subset of the dataset.



In [ ]:
model.fit(X_train, y_train)

14. Test the model using the testing subset of the dataset. Calculate and report the accuracy.



In [ ]:
predictors = model.predict(X_test)
predictors

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy = accuracy_score(y_test, predictors)

print("Accuracy:", accuracy)

###Cross Validation (CV)

15. Perform a cross-validation loop to calculate the accuracy of your model. Report that accuracy. How does it compare to the accuracy you calculated in #14?



In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
n = 100
results = np.zeros(n)

for i in range(n):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        stratify=y
    )

    model = GaussianNB()
    model.fit(X_train, y_train)

    predictors = model.predict(X_test)

    results[i] = metrics.accuracy_score(y_test, predictors)

print(results.mean())

###CV Histogram

16. Plot a histogram of the accuracy scores you generated in your cross-validation loop. What do you notice about the distribution of accuracy scores?



In [ ]:
plt.hist(results, bins=10, color='lightblue', edgecolor='black')
plt.title('Cross Validation Accuracy')
plt.xlabel("Accuracy")
plt.ylabel("Frequency")

plt.savefig('CV.png')
plt.show()

17.  Present the confusion matrix and the results of your Classification Report (sklearn.metrics.classification_report). What do you notice?



18. The training data is very skewed towards non-successful transactions (about 90% of the training data has ‘target’==0). Remove enough non-successful transaction rows so that your remaining training data is 50%/50% split between successful and non-successful transactions. Hint: you can use the data frames you created in step #9.



19. Repeat the cross-validation process on this data set. Report what your cross-validation accuracy is in this 50/50 case.

## Data Visualization


20. Compare the results of your cross-validation with the whole training data and the reduced 50/50 training data

1. Present the confusion matrix and the results of your Classification Report (sklearn.metrics.classification_report)




## Communicate the Results

22. Communicate the results of your analysis.



## Submit Final Project

23. Upload your finished Jupyter notebook to your Project 1 student folder.
